In [ ]:
---
layout: post
toc: true
title: Trimester 2 Security Growth
permalink: /security
author: Mihir Bapat
---

# 401, Builds, Testing, and Security: What This Tri Taught Us

This Tri gave us a clear reminder that engineering discipline is not separate from security. It is security.

When an API returns `401 Unauthorized`, that is not automatically a bug. In this codebase, a `401` usually means the endpoint is protected as designed, and the caller is missing the required authentication or role. The right response is to verify intent and policy, not to weaken security globally just to make a request pass.

At the same time, we had PRs that did not build and had to be pulled/reverted. That is not just a quality issue; it slows everyone down and creates avoidable risk. If code cannot build and test locally, it is not ready for review.

## The standard we need

Before a PR is submitted, the baseline is simple:

```bash
./mvnw clean compile
./mvnw test
```

If either command fails, fix locally first. Re-run. Then push.

## Breaking PRs and schema changes

Build failures are not the only blocker. If your PR changes schema or model structure, you must also prove database init/migration flow still works.

Required check for schema-related PRs:

```bash
python scripts/db_init.py
```

This command should run with **no errors** before requesting review. If it fails, your PR is not ready.

We also need to check security impact for endpoint changes. If you touched routes, confirm policy in:

- `src/main/java/com/open/spring/security/SecurityConfig.java`
- `src/main/java/com/open/spring/security/MvcSecurityConfig.java`

Least privilege should be the default: only open what must be open.

## 401 is a signal, not noise

A lot of confusion comes from interpreting `401` as “API is broken.” Usually it means one of these:

- missing authentication token/session
- wrong role for that endpoint
- endpoint intentionally restricted and not yet allowlisted

The fix is to confirm who should access the endpoint, then apply the minimal policy change needed.

## How this security system works right now

This project uses two security lanes:

1. **API lane** (`/api/**` and `/authenticate`) in `SecurityConfig`
   - JWT-based, stateless policy
   - Used by API clients and front-end API calls
2. **MVC lane** (`/**`) in `MvcSecurityConfig`
   - Form login + session behavior
   - Used by server-rendered pages

Because `SecurityConfig` has higher order, API requests are evaluated there first. If your endpoint starts with `/api/`, it should be configured in `SecurityConfig` (and optionally reinforced at the controller method level with `@PreAuthorize`).

### What happens on a protected API request

- Client sends request to `/api/...`
- JWT filter checks auth cookie/token
- Spring matches endpoint + HTTP method against role rules in `SecurityConfig`
- If no matching rule grants access, request is denied (`401`/`403` depending on state)

That means your endpoint can be perfectly implemented but still blocked until policy is updated intentionally.

## Example: adding a new API that allows ROLE_USER

Imagine we add this endpoint: `GET /api/weather/forecast`

### Step 1: Add the endpoint

```java
@RestController
@RequestMapping("/api/weather")
public class WeatherApiController {

	@GetMapping("/forecast")
	public ResponseEntity<Map<String, Object>> forecast() {
		Map<String, Object> body = Map.of(
			"summary", "Sunny",
			"high", 73,
			"low", 58
		);
		return ResponseEntity.ok(body);
	}
}
```

### Step 2: Allow access in `SecurityConfig`

```java
.requestMatchers(HttpMethod.GET, "/api/weather/forecast")
	.hasAnyAuthority("ROLE_USER", "ROLE_STUDENT", "ROLE_TEACHER", "ROLE_ADMIN")
```

Add this above broad fallback rules so it is explicit and easy to read.

### Step 3 (recommended): add method-level guard too

```java
@PreAuthorize("hasAnyAuthority('ROLE_USER','ROLE_STUDENT','ROLE_TEACHER','ROLE_ADMIN')")
@GetMapping("/forecast")
public ResponseEntity<Map<String, Object>> forecast() { ... }
```

Now both config-level and method-level policy agree. This gives defense in depth.

## Testing security before opening a PR

When you add or change an endpoint, test all three states:

1. **No auth** → should fail on protected endpoint
2. **Wrong role** → should fail
3. **Correct role** → should pass

### Quick local test pattern

```bash
# 1) Build + test first
./mvnw clean compile
./mvnw test

# 2) Hit endpoint without auth (expect 401/403/redirect based on endpoint type)
curl -i http://127.0.0.1:8585/api/weather/forecast

# 3) Authenticate and retry with user cookie/session
# (use your project's existing auth flow)
```

For role testing, use separate accounts (for example: one with `ROLE_USER`, one with admin). Record expected vs actual status codes in your PR notes.

## Security review checklist for students

Before requesting PR, verify:

- endpoint path and HTTP method are explicitly mapped in security config
- least-privilege role list is used (not over-broad)
- method-level guard exists for sensitive operations
- no accidental `permitAll` on write endpoints
- tests include unauthorized + authorized cases

This is how we avoid “it works locally” while still shipping secure APIs.

## Real-data testing and credential boundaries

There is a valid concern here: real-data testing often requires production pull access, which currently means admin-level credentials. Giving root-like credentials to the entire class is not acceptable.

So the practical approach for now is:

- students test with default local users/data initialized through:

```bash
python scripts/db_init.py
```

- instructor/admin handles production data pulls when required

The process is still transferable even without production credentials. The steps do not change; only the data source changes.

In the future, we may add a safer path where **user authentication + a scoped key** allows pulling real user data for testing without sharing root/admin credentials broadly.

## What we should improve next

Long term, we should build a safer middle path:

- user login + scoped key for data access
- read-limited permissions
- short-lived credentials with audit logs

That would let more people validate against realistic data without exposing privileged accounts.

Reliable builds, intentional security, and disciplined testing are the foundation. If we keep those tight, everything downstream gets easier.
